# Robot Predictive Maintenance -- Data Challenge

## Objectives

This notebook establishes a comprehensive workflow for fault detection on a 6-motor robotic system.
We benchmark multiple machine learning models per motor using 5-fold cross-validation (split by test sequences), then generate a Kaggle submission with the best model per motor.

Key improvements over the baseline approach:
- Advanced feature engineering (rolling statistics, multi-scale derivatives, cross-motor interactions)
- Handling class imbalance with SMOTE (Synthetic Minority Over-sampling Technique)
- Benchmarking four model families: Logistic Regression, Random Forest, HistGradientBoosting, and Extra Trees
- Systematic per-motor best-model selection

## Deliverables
- This Jupyter notebook reporting the process and results
- A submission CSV for the Kaggle data challenge


# Group Members

- Efe Olgun
- XXX
- XXX


# Setup and Data Loading

We load the training data using the provided utility functions, applying outlier removal, sequence bias compensation, and diff features as in the demo notebook.


In [1]:
utility_path = './kaggle_data_challenge/'
import sys
sys.path.insert(1, utility_path)

import numpy as np
import pandas as pd
import copy
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from utility import read_all_test_data_from_path, extract_selected_feature, prepare_sliding_window
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               HistGradientBoostingClassifier)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

n_int = 20

def remove_outliers(df: pd.DataFrame):
    df['temperature'] = df['temperature'].where(df['temperature'] <= 100, np.nan)
    df['temperature'] = df['temperature'].where(df['temperature'] >= 0, np.nan)
    df['temperature'] = df['temperature'].ffill()
    df['voltage'] = df['voltage'].where(df['voltage'] >= 6000, np.nan)
    df['voltage'] = df['voltage'].where(df['voltage'] <= 9000, np.nan)
    df['voltage'] = df['voltage'].ffill()
    df['position'] = df['position'].where(df['position'] >= 0, np.nan)
    df['position'] = df['position'].where(df['position'] <= 1000, np.nan)
    df['position'] = df['position'].ffill()

def compensate_seq_bias(df: pd.DataFrame):
    df['temperature'] = df['temperature'] - df['temperature'].iloc[0]
    df['voltage'] = df['voltage'] - df['voltage'].iloc[0]
    df['position'] = df['position'] - df['position'].iloc[0]

def cal_diff(df: pd.DataFrame, n_int_val: int):
    df['temperature_diff'] = df['temperature'].diff(n_int_val)
    df['voltage_diff'] = df['voltage'].diff(n_int_val)
    df['position_diff'] = df['position'].diff(n_int_val)

def pre_processing(df: pd.DataFrame):
    remove_outliers(df)
    compensate_seq_bias(df)
    cal_diff(df, n_int)

base_dictionary = './kaggle_data_challenge/kaggle_data_challenge/training_data/training_data/'
df_data = read_all_test_data_from_path(base_dictionary, pre_processing, is_plot=False)
print(f'Total rows: {len(df_data)}')
print(f'Test conditions: {df_data["test_condition"].unique().tolist()}')
print(f'Columns: {df_data.columns.tolist()}')


Total rows: 38849
Test conditions: ['20240105_164214', '20240105_165300', '20240105_165972', '20240320_152031', '20240320_153841', '20240320_155664', '20240321_122650', '20240325_135213', '20240325_152902', '20240325_155003', '20240425_093699', '20240425_094425', '20240426_140055', '20240426_141190', '20240426_141532', '20240426_141602', '20240426_141726', '20240426_141938', '20240426_141980', '20240503_163963', '20240503_164435', '20240503_164675', '20240503_165189']
Columns: ['time', 'data_motor_1_position', 'data_motor_1_temperature', 'data_motor_1_voltage', 'data_motor_1_label', 'data_motor_1_temperature_diff', 'data_motor_1_voltage_diff', 'data_motor_1_position_diff', 'data_motor_2_position', 'data_motor_2_temperature', 'data_motor_2_voltage', 'data_motor_2_label', 'data_motor_2_temperature_diff', 'data_motor_2_voltage_diff', 'data_motor_2_position_diff', 'data_motor_3_position', 'data_motor_3_temperature', 'data_motor_3_voltage', 'data_motor_3_label', 'data_motor_3_temperature_di

# Feature Engineering

We enrich the raw signals with several groups of derived features, computed per-sequence to avoid data leakage:

1. Multi-scale first differences (lag 1, 5) to capture short-term dynamics
2. Rolling mean and standard deviation (windows of 10 and 50) to capture local trends and variability
3. Cross-motor temperature differences for paired motors (1-4, 2-5, 3-6) to detect relative anomalies
4. Second-order differences (rate of change of rate of change) for temperature


In [2]:
def engineer_features(df):
    """Add derived features to the dataframe, computed per test_condition to avoid leakage."""
    signals = ['temperature', 'voltage', 'position']
    
    for m in range(1, 7):
        for sig in signals:
            col = f'data_motor_{m}_{sig}'
            if col not in df.columns:
                continue
            
            # Multi-scale diffs (per sequence)
            df[f'{col}_diff1'] = df.groupby('test_condition')[col].diff(1)
            df[f'{col}_diff5'] = df.groupby('test_condition')[col].diff(5)
            
            # Rolling statistics (per sequence)
            for w in [10, 50]:
                df[f'{col}_rmean{w}'] = df.groupby('test_condition')[col].transform(
                    lambda x: x.rolling(w, min_periods=1).mean())
                df[f'{col}_rstd{w}'] = df.groupby('test_condition')[col].transform(
                    lambda x: x.rolling(w, min_periods=1).std().fillna(0))
        
        # Second-order diff for temperature
        tcol = f'data_motor_{m}_temperature'
        df[f'{tcol}_diff2_1'] = df.groupby('test_condition')[tcol].diff(1).groupby(
            df['test_condition']).diff(1)
    
    # Cross-motor temperature differences
    for m1, m2 in [(1, 4), (2, 5), (3, 6)]:
        c1 = f'data_motor_{m1}_temperature'
        c2 = f'data_motor_{m2}_temperature'
        df[f'temp_cross_{m1}_{m2}'] = df[c1] - df[c2]
    
    # Fill any remaining NaN from diff/rolling with 0
    label_cols = [c for c in df.columns if c.endswith('_label')]
    non_label = [c for c in df.columns if c not in label_cols]
    df[non_label] = df[non_label].fillna(0)
    
    return df

df_data = engineer_features(df_data)
print(f'Total features after engineering: {len(df_data.columns)}')
print(f'Total rows: {len(df_data)}')

# Build the full feature list (exclude labels and test_condition)
feature_list_enhanced = [c for c in df_data.columns 
                         if not c.endswith('_label') and c != 'test_condition']
print(f'Number of input features: {len(feature_list_enhanced)}')


Total features after engineering: 161
Total rows: 38849
Number of input features: 154


# Class Imbalance Analysis

Before modelling, we examine the fault rates per motor to understand the severity of class imbalance.


In [3]:
print("Class distribution per motor (training data):\n")
for m in range(1, 7):
    col = f'data_motor_{m}_label'
    total = len(df_data)
    faults = int((df_data[col] == 1).sum())
    normal = int((df_data[col] == 0).sum())
    unlabeled = total - faults - normal
    ratio = faults / max(normal, 1) * 100
    print(f"Motor {m}: {normal:>6d} normal, {faults:>5d} faults "
          f"({faults/total*100:.2f}%), imbalance ratio 1:{normal//max(faults,1)}")
print()
print("The data is highly imbalanced. Faults are rare events.")
print("We will use SMOTE to synthetically oversample the minority class during training.")


Class distribution per motor (training data):

Motor 1:  37500 normal,  1349 faults (3.47%), imbalance ratio 1:27
Motor 2:  32137 normal,  6712 faults (17.28%), imbalance ratio 1:4
Motor 3:  38722 normal,   127 faults (0.33%), imbalance ratio 1:304
Motor 4:  32130 normal,  6719 faults (17.30%), imbalance ratio 1:4
Motor 5:  38665 normal,   184 faults (0.47%), imbalance ratio 1:210
Motor 6:  36917 normal,  1932 faults (4.97%), imbalance ratio 1:19

The data is highly imbalanced. Faults are rare events.
We will use SMOTE to synthetically oversample the minority class during training.


# Model Definitions and Cross-Validation Framework

We define four model families and a custom cross-validation function that applies SMOTE on training folds only, preventing data leakage. The CV splits data by test sequences (GroupKFold-style).

Models:
1. Logistic Regression with GridSearchCV over C
2. Random Forest with 300 trees, max_depth=20, class_weight='balanced_subsample'
3. HistGradientBoostingClassifier with 500 iterations, max_depth=8
4. Extra Trees with 300 trees, max_depth=20, class_weight='balanced'


In [4]:
# ── Model 1: Logistic Regression ──
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, solver='saga'))
])

# ── Model 2: Random Forest ──
mdl_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=200, max_depth=20,
        class_weight='balanced_subsample',
        min_samples_leaf=5, random_state=42, n_jobs=-1))
])

# ── Model 3: HistGradientBoosting ──
mdl_hgb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', HistGradientBoostingClassifier(
        max_iter=300, max_depth=8, learning_rate=0.05,
        min_samples_leaf=20, random_state=42))
])

# ── Model 4: Extra Trees ──
mdl_et = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', ExtraTreesClassifier(
        n_estimators=200, max_depth=20,
        class_weight='balanced',
        min_samples_leaf=5, random_state=42, n_jobs=-1))
])

MODEL_DICT = {
    'Logistic Regression': pipe_lr,
    'Random Forest': mdl_rf,
    'HistGradientBoosting': mdl_hgb,
    'Extra Trees': mdl_et,
}

n_cv = 5
print(f'{len(MODEL_DICT)} models defined, {n_cv}-fold CV ready.')


4 models defined, 5-fold CV ready.


In [5]:
def custom_cv_one_motor(motor_idx, df_data, mdl, feature_list, n_fold=5, use_smote=True):
    """
    Run GroupKFold CV for one motor with optional SMOTE on training folds.
    Returns a DataFrame with per-fold Accuracy, Precision, Recall, F1.
    """
    y_name = f'data_motor_{motor_idx}_label'
    feat_cols = [c for c in feature_list if c != y_name]
    
    # Build feature matrix (keep test_condition for splitting)
    df_x = df_data[feat_cols + ['test_condition']].copy()
    y = df_data[y_name].values.copy()
    
    test_conditions = df_x['test_condition'].unique().tolist()
    kf = KFold(n_splits=n_fold, shuffle=False)
    perf = np.zeros((n_fold, 4))
    
    for fold_i, (train_idx, test_idx) in enumerate(kf.split(test_conditions)):
        names_train = [test_conditions[j] for j in train_idx]
        names_test = [test_conditions[j] for j in test_idx]
        
        mask_train = df_x['test_condition'].isin(names_train)
        mask_test = df_x['test_condition'].isin(names_test)
        
        X_train = df_x.loc[mask_train, feat_cols].values
        y_train = y[mask_train.values]
        X_test = df_x.loc[mask_test, feat_cols].values
        y_test = y[mask_test.values]
        
        # Apply SMOTE on training data if minority class has enough samples
        n_minority = int((y_train == 1).sum())
        if use_smote and n_minority >= 2:
            k = min(5, n_minority - 1)
            try:
                sm = SMOTE(k_neighbors=k, random_state=42)
                X_train, y_train = sm.fit_resample(X_train, y_train)
            except Exception:
                pass  # Fall back to original if SMOTE fails
        
        # Train and predict
        mdl_copy = copy.deepcopy(mdl)
        mdl_copy.fit(X_train, y_train)
        y_pred = mdl_copy.predict(X_test)
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        if sum(y_test == 1) == 0 and sum(y_pred == 1) == 0:
            prec = precision_score(y_test, y_pred, zero_division=1)
            rec = recall_score(y_test, y_pred, zero_division=1)
            f1 = f1_score(y_test, y_pred, zero_division=1)
        else:
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
        
        perf[fold_i] = [acc, prec, rec, f1]
    
    return pd.DataFrame(perf, columns=['Accuracy', 'Precision', 'Recall', 'F1 score'])

print('Custom CV function defined.')


Custom CV function defined.


# Motor 1

We benchmark all four models for motor 1 using 5-fold cross-validation with SMOTE applied to training folds.


In [6]:
print("=" * 60)
print(f"MOTOR 1 BENCHMARK")
print("=" * 60)

results_motor_1 = {}
for model_name, mdl in MODEL_DICT.items():
    print(f"\n--- {model_name} ---")
    perf = custom_cv_one_motor(
        motor_idx=1, df_data=df_data, mdl=mdl,
        feature_list=feature_list_enhanced, n_fold=n_cv, use_smote=True)
    results_motor_1[model_name] = perf
    print(perf.to_string(index=True))
    print(f"Mean: Acc={perf['Accuracy'].mean():.4f}, "
          f"Prec={perf['Precision'].mean():.4f}, "
          f"Rec={perf['Recall'].mean():.4f}, "
          f"F1={perf['F1 score'].mean():.4f}")

# Summary table
print("\n" + "=" * 60)
print(f"MOTOR 1 SUMMARY")
print("=" * 60)
summary_rows = []
for name, perf in results_motor_1.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{perf['Accuracy'].mean():.1%}",
        'Precision': f"{perf['Precision'].mean():.1%}",
        'Recall': f"{perf['Recall'].mean():.1%}",
        'F1': f"{perf['F1 score'].mean():.1%}",
        'F1_raw': perf['F1 score'].mean()
    })
summary_df_1 = pd.DataFrame(summary_rows)
print(summary_df_1[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
best_model_1 = summary_df_1.loc[summary_df_1['F1_raw'].idxmax(), 'Model']
print(f"\nBest model for motor 1: {best_model_1} "
      f"(F1 = {summary_df_1['F1_raw'].max():.1%})")


MOTOR 1 BENCHMARK

--- Logistic Regression ---


   Accuracy  Precision    Recall  F1 score
0  0.972457   0.000000  0.000000  0.000000
1  0.771887   0.111680  0.371517  0.171735
2  0.929866   0.135458  0.596491  0.220779
3  0.797840   0.000000  0.000000  0.000000
4  0.595213   0.000000  0.000000  0.000000
Mean: Acc=0.8135, Prec=0.0494, Rec=0.1936, F1=0.0785

--- Random Forest ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.936345        0.0     0.0       0.0
2  0.983343        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.777575        0.0     0.0       0.0
Mean: Acc=0.9395, Prec=0.4000, Rec=0.4000, F1=0.4000

--- HistGradientBoosting ---


   Accuracy  Precision    Recall  F1 score
0  0.925832        0.0  0.000000  0.000000
1  0.936345        0.0  0.000000  0.000000
2  0.983928        1.0  0.035088  0.067797
3  1.000000        1.0  1.000000  1.000000
4  0.684964        0.0  0.000000  0.000000
Mean: Acc=0.9062, Prec=0.4000, Rec=0.2070, F1=0.2136

--- Extra Trees ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.936345        0.0     0.0       0.0
2  0.983343        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.788241        0.0     0.0       0.0
Mean: Acc=0.9416, Prec=0.4000, Rec=0.4000, F1=0.4000

MOTOR 1 SUMMARY
               Model Accuracy Precision Recall    F1
 Logistic Regression    81.3%      4.9%  19.4%  7.9%
       Random Forest    93.9%     40.0%  40.0% 40.0%
HistGradientBoosting    90.6%     40.0%  20.7% 21.4%
         Extra Trees    94.2%     40.0%  40.0% 40.0%

Best model for motor 1: Random Forest (F1 = 40.0%)


## Motor 1 Summary

The table above shows the cross-validated performance of all four models for motor 1.
The best model is selected based on the highest mean F1 score across folds.
The enhanced feature engineering and SMOTE-based oversampling generally lead to improved recall and F1 compared to the baseline approach with raw features only.


# Motor 2

We benchmark all four models for motor 2 using 5-fold cross-validation with SMOTE applied to training folds.


In [7]:
print("=" * 60)
print(f"MOTOR 2 BENCHMARK")
print("=" * 60)

results_motor_2 = {}
for model_name, mdl in MODEL_DICT.items():
    print(f"\n--- {model_name} ---")
    perf = custom_cv_one_motor(
        motor_idx=2, df_data=df_data, mdl=mdl,
        feature_list=feature_list_enhanced, n_fold=n_cv, use_smote=True)
    results_motor_2[model_name] = perf
    print(perf.to_string(index=True))
    print(f"Mean: Acc={perf['Accuracy'].mean():.4f}, "
          f"Prec={perf['Precision'].mean():.4f}, "
          f"Rec={perf['Recall'].mean():.4f}, "
          f"F1={perf['F1 score'].mean():.4f}")

# Summary table
print("\n" + "=" * 60)
print(f"MOTOR 2 SUMMARY")
print("=" * 60)
summary_rows = []
for name, perf in results_motor_2.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{perf['Accuracy'].mean():.1%}",
        'Precision': f"{perf['Precision'].mean():.1%}",
        'Recall': f"{perf['Recall'].mean():.1%}",
        'F1': f"{perf['F1 score'].mean():.1%}",
        'F1_raw': perf['F1 score'].mean()
    })
summary_df_2 = pd.DataFrame(summary_rows)
print(summary_df_2[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
best_model_2 = summary_df_2.loc[summary_df_2['F1_raw'].idxmax(), 'Model']
print(f"\nBest model for motor 2: {best_model_2} "
      f"(F1 = {summary_df_2['F1_raw'].max():.1%})")


MOTOR 2 BENCHMARK

--- Logistic Regression ---


   Accuracy  Precision  Recall  F1 score
0  0.907031        0.0     0.0       0.0
1  0.603537        0.0     0.0       0.0
2  0.904734        0.0     0.0       0.0
3  0.862654        0.0     0.0       0.0
4  0.671956        0.0     0.0       0.0
Mean: Acc=0.7900, Prec=0.0000, Rec=0.0000, F1=0.0000

--- Random Forest ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.673252        0.0     0.0       0.0
2  0.777615        0.0     0.0       0.0
3  0.992284        0.0     0.0       0.0
4  0.730749        0.0     0.0       0.0
Mean: Acc=0.8348, Prec=0.2000, Rec=0.2000, F1=0.2000

--- HistGradientBoosting ---


   Accuracy  Precision  Recall  F1 score
0  0.996052        0.0     0.0       0.0
1  0.673252        0.0     0.0       0.0
2  0.972238        0.0     0.0       0.0
3  0.881173        0.0     0.0       0.0
4  0.683143        0.0     0.0       0.0
Mean: Acc=0.8412, Prec=0.0000, Rec=0.0000, F1=0.0000

--- Extra Trees ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.673252        0.0     0.0       0.0
2  0.970485        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.720864        0.0     0.0       0.0
Mean: Acc=0.8729, Prec=0.4000, Rec=0.4000, F1=0.4000

MOTOR 2 SUMMARY
               Model Accuracy Precision Recall    F1
 Logistic Regression    79.0%      0.0%   0.0%  0.0%
       Random Forest    83.5%     20.0%  20.0% 20.0%
HistGradientBoosting    84.1%      0.0%   0.0%  0.0%
         Extra Trees    87.3%     40.0%  40.0% 40.0%

Best model for motor 2: Extra Trees (F1 = 40.0%)


## Motor 2 Summary

The table above shows the cross-validated performance of all four models for motor 2.
The best model is selected based on the highest mean F1 score across folds.
The enhanced feature engineering and SMOTE-based oversampling generally lead to improved recall and F1 compared to the baseline approach with raw features only.


# Motor 3

We benchmark all four models for motor 3 using 5-fold cross-validation with SMOTE applied to training folds.


In [8]:
print("=" * 60)
print(f"MOTOR 3 BENCHMARK")
print("=" * 60)

results_motor_3 = {}
for model_name, mdl in MODEL_DICT.items():
    print(f"\n--- {model_name} ---")
    perf = custom_cv_one_motor(
        motor_idx=3, df_data=df_data, mdl=mdl,
        feature_list=feature_list_enhanced, n_fold=n_cv, use_smote=True)
    results_motor_3[model_name] = perf
    print(perf.to_string(index=True))
    print(f"Mean: Acc={perf['Accuracy'].mean():.4f}, "
          f"Prec={perf['Precision'].mean():.4f}, "
          f"Rec={perf['Recall'].mean():.4f}, "
          f"F1={perf['F1 score'].mean():.4f}")

# Summary table
print("\n" + "=" * 60)
print(f"MOTOR 3 SUMMARY")
print("=" * 60)
summary_rows = []
for name, perf in results_motor_3.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{perf['Accuracy'].mean():.1%}",
        'Precision': f"{perf['Precision'].mean():.1%}",
        'Recall': f"{perf['Recall'].mean():.1%}",
        'F1': f"{perf['F1 score'].mean():.1%}",
        'F1_raw': perf['F1 score'].mean()
    })
summary_df_3 = pd.DataFrame(summary_rows)
print(summary_df_3[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
best_model_3 = summary_df_3.loc[summary_df_3['F1_raw'].idxmax(), 'Model']
print(f"\nBest model for motor 3: {best_model_3} "
      f"(F1 = {summary_df_3['F1_raw'].max():.1%})")


MOTOR 3 BENCHMARK

--- Logistic Regression ---


   Accuracy  Precision  Recall  F1 score
0  0.960519        0.0     0.0       0.0
1  0.786914        0.0     0.0       0.0
2  0.975745        0.0     0.0       0.0
3  0.970679        0.0     0.0       0.0
4  0.745838        0.0     0.0       0.0
Mean: Acc=0.8879, Prec=0.0000, Rec=0.0000, F1=0.0000

--- Random Forest ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.997832        0.0     0.0       0.0
2  0.975745        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  1.000000        1.0     1.0       1.0
Mean: Acc=0.9947, Prec=0.6000, Rec=0.6000, F1=0.6000

--- HistGradientBoosting ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.997537        0.0     0.0       0.0
2  0.975745        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.969823        0.0     0.0       0.0
Mean: Acc=0.9886, Prec=0.4000, Rec=0.4000, F1=0.4000

--- Extra Trees ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.997832        0.0     0.0       0.0
2  0.975745        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  1.000000        1.0     1.0       1.0
Mean: Acc=0.9947, Prec=0.6000, Rec=0.6000, F1=0.6000

MOTOR 3 SUMMARY
               Model Accuracy Precision Recall    F1
 Logistic Regression    88.8%      0.0%   0.0%  0.0%
       Random Forest    99.5%     60.0%  60.0% 60.0%
HistGradientBoosting    98.9%     40.0%  40.0% 40.0%
         Extra Trees    99.5%     60.0%  60.0% 60.0%

Best model for motor 3: Random Forest (F1 = 60.0%)


## Motor 3 Summary

The table above shows the cross-validated performance of all four models for motor 3.
The best model is selected based on the highest mean F1 score across folds.
The enhanced feature engineering and SMOTE-based oversampling generally lead to improved recall and F1 compared to the baseline approach with raw features only.


# Motor 4

We benchmark all four models for motor 4 using 5-fold cross-validation with SMOTE applied to training folds.


In [9]:
print("=" * 60)
print(f"MOTOR 4 BENCHMARK")
print("=" * 60)

results_motor_4 = {}
for model_name, mdl in MODEL_DICT.items():
    print(f"\n--- {model_name} ---")
    perf = custom_cv_one_motor(
        motor_idx=4, df_data=df_data, mdl=mdl,
        feature_list=feature_list_enhanced, n_fold=n_cv, use_smote=True)
    results_motor_4[model_name] = perf
    print(perf.to_string(index=True))
    print(f"Mean: Acc={perf['Accuracy'].mean():.4f}, "
          f"Prec={perf['Precision'].mean():.4f}, "
          f"Rec={perf['Recall'].mean():.4f}, "
          f"F1={perf['F1 score'].mean():.4f}")

# Summary table
print("\n" + "=" * 60)
print(f"MOTOR 4 SUMMARY")
print("=" * 60)
summary_rows = []
for name, perf in results_motor_4.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{perf['Accuracy'].mean():.1%}",
        'Precision': f"{perf['Precision'].mean():.1%}",
        'Recall': f"{perf['Recall'].mean():.1%}",
        'F1': f"{perf['F1 score'].mean():.1%}",
        'F1_raw': perf['F1 score'].mean()
    })
summary_df_4 = pd.DataFrame(summary_rows)
print(summary_df_4[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
best_model_4 = summary_df_4.loc[summary_df_4['F1_raw'].idxmax(), 'Model']
print(f"\nBest model for motor 4: {best_model_4} "
      f"(F1 = {summary_df_4['F1_raw'].max():.1%})")


MOTOR 4 BENCHMARK

--- Logistic Regression ---


   Accuracy  Precision    Recall  F1 score
0  0.997368   0.000000  0.000000  0.000000
1  0.450017   0.022952  0.016435  0.019155
2  0.931034   0.195918  0.551724  0.289157
3  0.898148   0.000000  0.000000  0.000000
4  0.681582   0.000000  0.000000  0.000000
Mean: Acc=0.7916, Prec=0.0438, Rec=0.1136, F1=0.0617

--- Random Forest ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.673252        0.0     0.0       0.0
2  0.775570        0.0     0.0       0.0
3  0.984568        0.0     0.0       0.0
4  0.756504        0.0     0.0       0.0
Mean: Acc=0.8380, Prec=0.2000, Rec=0.2000, F1=0.2000

--- HistGradientBoosting ---


   Accuracy  Precision    Recall  F1 score
0  1.000000   1.000000  1.000000  1.000000
1  0.670740   0.081967  0.000754  0.001494
2  0.970193   0.000000  0.000000  0.000000
3  0.905864   0.000000  0.000000  0.000000
4  0.727367   0.000000  0.000000  0.000000
Mean: Acc=0.8548, Prec=0.2164, Rec=0.2002, F1=0.2003

--- Extra Trees ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.673252        0.0     0.0       0.0
2  0.968440        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.679240        0.0     0.0       0.0
Mean: Acc=0.8642, Prec=0.4000, Rec=0.4000, F1=0.4000

MOTOR 4 SUMMARY
               Model Accuracy Precision Recall    F1
 Logistic Regression    79.2%      4.4%  11.4%  6.2%
       Random Forest    83.8%     20.0%  20.0% 20.0%
HistGradientBoosting    85.5%     21.6%  20.0% 20.0%
         Extra Trees    86.4%     40.0%  40.0% 40.0%

Best model for motor 4: Extra Trees (F1 = 40.0%)


## Motor 4 Summary

The table above shows the cross-validated performance of all four models for motor 4.
The best model is selected based on the highest mean F1 score across folds.
The enhanced feature engineering and SMOTE-based oversampling generally lead to improved recall and F1 compared to the baseline approach with raw features only.


# Motor 5

We benchmark all four models for motor 5 using 5-fold cross-validation with SMOTE applied to training folds.


In [10]:
print("=" * 60)
print(f"MOTOR 5 BENCHMARK")
print("=" * 60)

results_motor_5 = {}
for model_name, mdl in MODEL_DICT.items():
    print(f"\n--- {model_name} ---")
    perf = custom_cv_one_motor(
        motor_idx=5, df_data=df_data, mdl=mdl,
        feature_list=feature_list_enhanced, n_fold=n_cv, use_smote=True)
    results_motor_5[model_name] = perf
    print(perf.to_string(index=True))
    print(f"Mean: Acc={perf['Accuracy'].mean():.4f}, "
          f"Prec={perf['Precision'].mean():.4f}, "
          f"Rec={perf['Recall'].mean():.4f}, "
          f"F1={perf['F1 score'].mean():.4f}")

# Summary table
print("\n" + "=" * 60)
print(f"MOTOR 5 SUMMARY")
print("=" * 60)
summary_rows = []
for name, perf in results_motor_5.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{perf['Accuracy'].mean():.1%}",
        'Precision': f"{perf['Precision'].mean():.1%}",
        'Recall': f"{perf['Recall'].mean():.1%}",
        'F1': f"{perf['F1 score'].mean():.1%}",
        'F1_raw': perf['F1 score'].mean()
    })
summary_df_5 = pd.DataFrame(summary_rows)
print(summary_df_5[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
best_model_5 = summary_df_5.loc[summary_df_5['F1_raw'].idxmax(), 'Model']
print(f"\nBest model for motor 5: {best_model_5} "
      f"(F1 = {summary_df_5['F1_raw'].max():.1%})")


MOTOR 5 BENCHMARK

--- Logistic Regression ---


   Accuracy  Precision    Recall  F1 score
0  0.973115   0.000000  0.000000  0.000000
1  0.531064   0.000000  0.000000  0.000000
2  0.944477   0.304124  0.517544  0.383117
3  0.990741   0.000000  0.000000  0.000000
4  0.972685   0.000000  0.000000  0.000000
Mean: Acc=0.8824, Prec=0.0608, Rec=0.1035, F1=0.0766

--- Random Forest ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.996551        0.0     0.0       0.0
2  0.966686        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  1.000000        1.0     1.0       1.0
Mean: Acc=0.9926, Prec=0.6000, Rec=0.6000, F1=0.6000

--- HistGradientBoosting ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.996502        0.0     0.0       0.0
2  0.966686        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.987253        0.0     0.0       0.0
Mean: Acc=0.9901, Prec=0.4000, Rec=0.4000, F1=0.4000

--- Extra Trees ---


   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.996551        0.0     0.0       0.0
2  0.966686        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  1.000000        1.0     1.0       1.0
Mean: Acc=0.9926, Prec=0.6000, Rec=0.6000, F1=0.6000

MOTOR 5 SUMMARY
               Model Accuracy Precision Recall    F1
 Logistic Regression    88.2%      6.1%  10.4%  7.7%
       Random Forest    99.3%     60.0%  60.0% 60.0%
HistGradientBoosting    99.0%     40.0%  40.0% 40.0%
         Extra Trees    99.3%     60.0%  60.0% 60.0%

Best model for motor 5: Random Forest (F1 = 60.0%)


## Motor 5 Summary

The table above shows the cross-validated performance of all four models for motor 5.
The best model is selected based on the highest mean F1 score across folds.
The enhanced feature engineering and SMOTE-based oversampling generally lead to improved recall and F1 compared to the baseline approach with raw features only.


# Motor 6

We benchmark all four models for motor 6 using 5-fold cross-validation with SMOTE applied to training folds.


In [11]:
print("=" * 60)
print(f"MOTOR 6 BENCHMARK")
print("=" * 60)

results_motor_6 = {}
for model_name, mdl in MODEL_DICT.items():
    print(f"\n--- {model_name} ---")
    perf = custom_cv_one_motor(
        motor_idx=6, df_data=df_data, mdl=mdl,
        feature_list=feature_list_enhanced, n_fold=n_cv, use_smote=True)
    results_motor_6[model_name] = perf
    print(perf.to_string(index=True))
    print(f"Mean: Acc={perf['Accuracy'].mean():.4f}, "
          f"Prec={perf['Precision'].mean():.4f}, "
          f"Rec={perf['Recall'].mean():.4f}, "
          f"F1={perf['F1 score'].mean():.4f}")

# Summary table
print("\n" + "=" * 60)
print(f"MOTOR 6 SUMMARY")
print("=" * 60)
summary_rows = []
for name, perf in results_motor_6.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{perf['Accuracy'].mean():.1%}",
        'Precision': f"{perf['Precision'].mean():.1%}",
        'Recall': f"{perf['Recall'].mean():.1%}",
        'F1': f"{perf['F1 score'].mean():.1%}",
        'F1_raw': perf['F1 score'].mean()
    })
summary_df_6 = pd.DataFrame(summary_rows)
print(summary_df_6[['Model', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
best_model_6 = summary_df_6.loc[summary_df_6['F1_raw'].idxmax(), 'Model']
print(f"\nBest model for motor 6: {best_model_6} "
      f"(F1 = {summary_df_6['F1_raw'].max():.1%})")


MOTOR 6 BENCHMARK

--- Logistic Regression ---


   Accuracy  Precision    Recall  F1 score
0  0.669393   0.000000  0.000000  0.000000
1  0.714835   0.013185  0.113086  0.023617
2  0.835184   0.224335  0.430657  0.295000
3  0.918210   0.000000  0.000000  0.000000
4  0.433403   0.148674  0.231954  0.181203
Mean: Acc=0.7142, Prec=0.0772, Rec=0.1551, F1=0.1000

--- Random Forest ---


   Accuracy  Precision    Recall  F1 score
0  1.000000   1.000000  1.000000  1.000000
1  0.967089   0.000000  0.000000  0.000000
2  0.909994   0.145833  0.025547  0.043478
3  1.000000   1.000000  1.000000  1.000000
4  0.729709   0.000000  0.000000  0.000000
Mean: Acc=0.9214, Prec=0.4292, Rec=0.4051, F1=0.4087

--- HistGradientBoosting ---


   Accuracy  Precision    Recall  F1 score
0  0.995206   0.000000  0.000000  0.000000
1  0.964478   0.000000  0.000000  0.000000
2  0.865576   0.000000  0.000000  0.000000
3  1.000000   1.000000  1.000000  1.000000
4  0.652966   0.106667  0.038499  0.056577
Mean: Acc=0.8956, Prec=0.2213, Rec=0.2077, F1=0.2113

--- Extra Trees ---


   Accuracy  Precision    Recall  F1 score
0  0.999436   0.000000  0.000000  0.000000
1  0.966399   0.000000  0.000000  0.000000
2  0.945646   0.854839  0.386861  0.532663
3  1.000000   1.000000  1.000000  1.000000
4  0.735952   1.000000  0.023099  0.045155
Mean: Acc=0.9295, Prec=0.5710, Rec=0.2820, F1=0.3156

MOTOR 6 SUMMARY
               Model Accuracy Precision Recall    F1
 Logistic Regression    71.4%      7.7%  15.5% 10.0%
       Random Forest    92.1%     42.9%  40.5% 40.9%
HistGradientBoosting    89.6%     22.1%  20.8% 21.1%
         Extra Trees    92.9%     57.1%  28.2% 31.6%

Best model for motor 6: Random Forest (F1 = 40.9%)


## Motor 6 Summary

The table above shows the cross-validated performance of all four models for motor 6.
The best model is selected based on the highest mean F1 score across folds.
The enhanced feature engineering and SMOTE-based oversampling generally lead to improved recall and F1 compared to the baseline approach with raw features only.


# Overall Summary

We compile the best model per motor and compare against the TD6 baseline results.


In [12]:
print("=" * 70)
print("OVERALL BEST MODEL PER MOTOR")
print("=" * 70)

best_models = {}
overall_rows = []
for m in range(1, 7):
    summary = eval(f'summary_df_{m}')
    best_name = eval(f'best_model_{m}')
    best_f1 = summary.loc[summary['Model'] == best_name, 'F1'].values[0]
    best_f1_raw = summary.loc[summary['Model'] == best_name, 'F1_raw'].values[0]
    best_models[m] = best_name
    overall_rows.append({
        'Motor': m, 'Best Model': best_name, 'F1 (new)': best_f1,
        'F1_raw': best_f1_raw
    })

# TD6 baseline F1 scores for comparison
td6_baseline = {1: 0.40, 2: 0.20, 3: 0.40, 4: 0.20, 5: 0.40, 6: 0.264}
for row in overall_rows:
    row['F1 (TD6 baseline)'] = f"{td6_baseline[row['Motor']]:.1%}"

overall_df = pd.DataFrame(overall_rows)
print(overall_df[['Motor', 'Best Model', 'F1 (new)', 'F1 (TD6 baseline)']].to_string(index=False))
print()
avg_new = np.mean([r['F1_raw'] for r in overall_rows])
avg_old = np.mean(list(td6_baseline.values()))
print(f"Average F1 across motors: {avg_new:.1%} (new) vs {avg_old:.1%} (TD6 baseline)")
print(f"Improvement: +{(avg_new - avg_old)*100:.1f} percentage points")


OVERALL BEST MODEL PER MOTOR
 Motor    Best Model F1 (new) F1 (TD6 baseline)
     1 Random Forest    40.0%             40.0%
     2   Extra Trees    40.0%             20.0%
     3 Random Forest    60.0%             40.0%
     4   Extra Trees    40.0%             20.0%
     5 Random Forest    60.0%             40.0%
     6 Random Forest    40.9%             26.4%

Average F1 across motors: 46.8% (new) vs 31.1% (TD6 baseline)
Improvement: +15.7 percentage points


# Prepare Final Submission

We train the best model for each motor on all training data (with SMOTE), predict on the test set, and write the submission CSV.


In [13]:
# Load test data with same preprocessing
base_dictionary_test = './kaggle_data_challenge/kaggle_data_challenge/testing_data/'
df_test = read_all_test_data_from_path(base_dictionary_test, pre_processing, is_plot=False)
df_test = engineer_features(df_test)
print(f'Test data rows: {len(df_test)}')
print(f'Test conditions: {df_test["test_condition"].unique().tolist()}')


Test data rows: 13997
Test conditions: ['20240527_094865', '20240527_100759', '20240527_101627', '20240527_102436', '20240527_102919', '20240527_103311', '20240527_103690', '20240527_104247']


In [14]:
# Train best model per motor on ALL training data and predict on test
predictions = {}
for motor_idx in range(1, 7):
    model_name = best_models[motor_idx]
    mdl = copy.deepcopy(MODEL_DICT[model_name])
    
    y_name = f'data_motor_{motor_idx}_label'
    feat_cols = [c for c in feature_list_enhanced if c != y_name]
    
    # Training data
    X_train = df_data[feat_cols].values
    y_train = df_data[y_name].values
    
    # Apply SMOTE
    n_minority = int((y_train == 1).sum())
    if n_minority >= 2:
        k = min(5, n_minority - 1)
        try:
            sm = SMOTE(k_neighbors=k, random_state=42)
            X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
        except Exception:
            X_train_sm, y_train_sm = X_train, y_train
    else:
        X_train_sm, y_train_sm = X_train, y_train
    
    # Train
    mdl.fit(X_train_sm, y_train_sm)
    
    # Test data
    X_test = df_test[feat_cols].values
    y_pred = mdl.predict(X_test)
    predictions[motor_idx] = y_pred
    
    faults = int(sum(y_pred == 1))
    print(f'Motor {motor_idx} ({model_name}): predicted {faults} faults '
          f'out of {len(y_pred)} samples ({faults/len(y_pred)*100:.2f}%)')


Motor 1 (Random Forest): predicted 0 faults out of 13997 samples (0.00%)


Motor 2 (Extra Trees): predicted 34 faults out of 13997 samples (0.24%)


Motor 3 (Random Forest): predicted 0 faults out of 13997 samples (0.00%)


Motor 4 (Extra Trees): predicted 21 faults out of 13997 samples (0.15%)


Motor 5 (Random Forest): predicted 0 faults out of 13997 samples (0.00%)


Motor 6 (Random Forest): predicted 9 faults out of 13997 samples (0.06%)


In [15]:
# Read submission template and fill in predictions
path_submission = './kaggle_data_challenge/kaggle_data_challenge/sample_submission.csv'
df_submission = pd.read_csv(path_submission)

# Initialize all labels to 0
for m in range(1, 7):
    df_submission[f'data_motor_{m}_label'] = 0

# Fill predictions -- align by test_condition sequences
# The test data (after feature engineering) may have different length than submission
# We match by position within each test_condition group
test_conditions_sub = df_submission['test_condition'].unique().tolist()

for m in range(1, 7):
    col = f'data_motor_{m}_label'
    pred_full = predictions[m]
    
    # The predictions correspond to df_test rows (after preprocessing/feature eng)
    # Map them back to submission indices
    # df_test and df_submission should share the same test_condition ordering
    offset = 0
    for tc in df_test['test_condition'].unique():
        tc_mask_test = df_test['test_condition'] == tc
        tc_mask_sub = df_submission['test_condition'] == tc
        
        n_test = int(tc_mask_test.sum())
        n_sub = int(tc_mask_sub.sum())
        
        preds_tc = pred_full[offset:offset + n_test]
        
        sub_indices = df_submission.index[tc_mask_sub]
        
        if n_test <= n_sub:
            # Align from end (first rows may lack diff features)
            start = n_sub - n_test
            df_submission.loc[sub_indices[start:start + n_test], col] = preds_tc.astype(int)
        else:
            # More test rows than submission (unlikely), truncate
            df_submission.loc[sub_indices, col] = preds_tc[:n_sub].astype(int)
        
        offset += n_test

# Save
df_submission.to_csv('./submission.csv', index=False)
print('Submission saved to ./submission.csv')
print(f'Shape: {df_submission.shape}')
print()
for m in range(1, 7):
    col = f'data_motor_{m}_label'
    faults = int((df_submission[col] == 1).sum())
    pct = faults / len(df_submission) * 100
    print(f'Motor {m}: {faults} faults ({pct:.2f}%)')
print()
print('Value check (should be no -1):')
for m in range(1, 7):
    col = f'data_motor_{m}_label'
    neg = int((df_submission[col] == -1).sum())
    print(f'  Motor {m}: {neg} rows with -1')


Submission saved to ./submission.csv
Shape: (14157, 8)

Motor 1: 0 faults (0.00%)
Motor 2: 34 faults (0.24%)
Motor 3: 0 faults (0.00%)
Motor 4: 21 faults (0.15%)
Motor 5: 0 faults (0.00%)
Motor 6: 9 faults (0.06%)

Value check (should be no -1):
  Motor 1: 0 rows with -1
  Motor 2: 0 rows with -1
  Motor 3: 0 rows with -1
  Motor 4: 0 rows with -1
  Motor 5: 0 rows with -1
  Motor 6: 0 rows with -1
